# 02 · Train SignNet (CNN tự thiết kế) trên crop 64x64

Kaggle GPU T4 (vài phút/cấu hình). Input: code + output notebook 01 (`crops.tgz`).
So 3 cách xử lý lệch lớp: CE thường / + sampler căn bậc 2 / + sampler + loss có trọng số.
Thước đo chính là **macro-F1** (mỗi lớp nặng như nhau), accuracy bị lớp đông áp đảo.

In [ ]:
import glob, json, os, sys

CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE
TGZ = glob.glob("/kaggle/input/**/crops.tgz", recursive=True)[0]
!tar xzf {TGZ}
!pip install -q onnx onnxruntime

In [ ]:
runs = {
    "ce": "--sampler none",
    "sqrt": "--sampler sqrt",
    "sqrt+weighted": "--sampler sqrt --weighted-loss",
}
for name, flags in runs.items():
    !python -m dashcam.train_classifier --crops datasets/vnts/crops --names datasets/vnts/names.json \
        --out runs/cls_{name} --epochs 40 {flags}

In [ ]:
import numpy as np, torch
from dashcam.classifier import SignNet
from dashcam.train_classifier import metrics, predict

names = json.load(open("datasets/vnts/names.json"))
stats = json.load(open("datasets/vnts/class_stats.json"))["train"]
te = np.load("datasets/vnts/crops/test.npz")
rare = {i for i, n in enumerate(names) if stats.get(n, 0) < 50}
dev = torch.device("cuda")
res = {}
for name in runs:
    st = torch.load(f"runs/cls_{name}/best.pt", map_location=dev)
    m = SignNet(len(names), st["width"]).to(dev); m.load_state_dict(st["model"])
    p = predict(m, te["x"], dev)
    mt = metrics(p, te["y"], len(names))
    rare_rec = np.mean([r for c, r in mt["per_class_recall"].items() if c in rare]) if rare else None
    res[name] = {"acc": round(mt["acc"], 4), "macro_f1": round(mt["macro_f1"], 4),
                 "recall_lớp_hiếm": None if rare_rec is None else round(float(rare_rec), 4)}
import pandas as pd
pd.DataFrame(res).T

## Nhầm lẫn hay gặp (cấu hình tốt nhất)

In [ ]:
best = max(res, key=lambda k: res[k]["macro_f1"])
st = torch.load(f"runs/cls_{best}/best.pt", map_location=dev)
m = SignNet(len(names), st["width"]).to(dev); m.load_state_dict(st["model"])
pred = predict(m, te["x"], dev).argmax(1)
from collections import Counter
conf = Counter((names[g], names[p]) for g, p in zip(te["y"], pred) if g != p)
pd.DataFrame([(g, p, n) for (g, p), n in conf.most_common(15)], columns=["đúng", "đoán", "số lần"])

In [ ]:
!mkdir -p models && cp runs/cls_{best}/best.pt runs/signnet_best.pt
!python -m dashcam.export --signnet runs/signnet_best.pt --calib-crops datasets/vnts/crops/train.npz --out models
json.dump(res, open("models/classifier_results.json", "w"), ensure_ascii=False, indent=1)